# 🚀 VIENEU-TTS V3 TURBO - GOOGLE COLAB T4 GPU WORKER
Notebook này kết nối Google Colab T4 GPU với **TTS Proxy Gateway** qua Reverse WebSocket.
- Tự động Mount Google Drive và Cache Model 2GB để **không phải tải lại** mỗi lần bật lại Colab.
- Hỗ trợ nạp trực tiếp core engine từ repository: `https://github.com/pnnbao97/VieNeu-TTS.git`
- Nhận tác vụ theo thời gian thực và đẩy file trực tiếp lên Supabase Bucket `tts-audios`.

In [ ]:
# BƯỚC 1: KIỂM TRA GPU NVIDIA T4
!nvidia-smi

In [ ]:
# BƯỚC 2: MOUNT GOOGLE DRIVE ĐỂ LƯU MODEL & CACHE VĨNH VIỄN
from google.colab import drive
import os

drive.mount('/content/drive')

# Thiết lập thư mục Cache cho Hugging Face & VieNeu trên Google Drive
CACHE_DIR = '/content/drive/MyDrive/VieNeu_TTS_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
os.environ['TORCH_HOME'] = CACHE_DIR
print(f"✅ Thư mục Cache trên Google Drive: {CACHE_DIR}")

In [ ]:
# BƯỚC 3: TẢI REPOSITORY VIENEU-TTS & CÀI ĐẶT THƯ VIỆN
# 1. Clone repository chính thức pnnbao97/VieNeu-TTS vào Drive hoặc Colab
if not os.path.exists('/content/VieNeu-TTS'):
    !git clone https://github.com/pnnbao97/VieNeu-TTS.git /content/VieNeu-TTS

# 2. Cài đặt các gói phụ thuộc (websockets, supabase, vieneu...)
!pip install -q vieneu websockets supabase soundfile pydub python-dotenv accelerate transformers
print("✅ Cài đặt môi trường VieNeu-TTS thành công!")

In [ ]:
# BƯỚC 4: THIẾT LẬP THÔNG SỐ KẾT NỐI GATEWAY VÀ SUPABASE
# HÃY ĐIỀN THÔNG SỐ CỦA BẠN VÀO DƯỚI ĐÂY:
os.environ['GATEWAY_URL'] = 'wss://your-backend-gateway.com'      # Địa chỉ Backend FastAPI Gateway
os.environ['WORKER_SECRET_KEY'] = 'vieneu_worker_secret_2026'     # Khóa bí mật Worker (khớp với backend/.env)
os.environ['SUPABASE_URL'] = 'https://your-project.supabase.co'    # URL Supabase Project
os.environ['SUPABASE_KEY'] = 'YOUR_SUPABASE_SERVICE_ROLE_KEY'     # Supabase Service Role Key
os.environ['SUPABASE_BUCKET'] = 'tts-audios'

print("✅ Cấu hình môi trường thành công! Sẵn sàng khởi động Worker.")

In [ ]:
# BƯỚC 5: KHỞI CHẠY WORKER LẮNG NGHE JOB THEO THỜI GIAN THỰC
!python /content/colab-worker/worker.py
# (Hoặc nếu đang mở thư mục colab-worker: !python worker.py)